# pyQuil Tutorial: The Wavefunction Simulator

*Condensed from the [pyQuil docs, v4.17](https://pyquil-docs.rigetti.com/en/stable/wavefunction_simulator.html).*

$\newcommand{\ket}[1]{\left|#1\right\rangle}\newcommand{\bra}[1]{\left\langle#1\right|}\newcommand{\tr}[1]{\mathrm{Tr}\left[#1\right]}$

The wavefunction simulator lets you inspect the wavefunction of the quantum state your program prepares. Because quantum information is probabilistic, programs on a QPU or QVM return a distribution of outcomes, and you'd aggregate many trials (tens to 100k+) to estimate it.

With the wavefunction simulator you can see the distribution directly, without sampling — a big time saver for small programs.

If you haven’t built the **Python (pyQuil container)** kernel and run the **quilc (quilc -P -S)** and **qvm (qvm -S)** servers yet, run the **0_Jupyter_Pykernel_from_Container_image** notebook to create it from pyquil.sif, then select **Python (pyQuil container)** in the top-right corner (refresh the page if it isn’t listed). Cells that call `get_qc()`, `compile()` or `run()` need the `quilc` and `qvm` servers running:

In [1]:
import warnings
from pyquil._deprecation import PyQuilDeprecationWarning
# Hide pyQuil 4.22's notices that the QVM is deprecated in favour of pyQuil v5
# (the QVM still works; see 0_Jupyter_Pykernel_from_Container_image.ipynb).
warnings.filterwarnings("ignore", category=PyQuilDeprecationWarning)

from pyquil import Program
from pyquil.gates import *
from pyquil.api import WavefunctionSimulator

wf_sim = WavefunctionSimulator()
coin_flip = Program(H(0))
wavefunction = wf_sim.wavefunction(coin_flip)

The return value is a `Wavefunction` object that stores the amplitudes of the quantum state. Printing it shows the amplitudes as a sum of computational basis states:

In [2]:
print(wavefunction)

(0.7071067812+0j)|0> + (0.7071067812+0j)|1>


Expected output: `(0.7071067812+0j)|0> + (0.7071067812+0j)|1>`

You can index into the amplitudes directly, or get a dictionary of outcome probabilities:

In [3]:
import numpy as np
np.set_printoptions(legacy="1.21")

assert wavefunction[0] == 1 / np.sqrt(2)
# The amplitudes are stored as a numpy array on the Wavefunction object
print(wavefunction.amplitudes)
prob_dict = wavefunction.get_outcome_probs()  # extracts the probabilities of outcomes as a dict
print(prob_dict)
print(prob_dict.keys())  # these store the bitstring outcomes
assert len(wavefunction) == 1  # gives the number of qubits

[0.70710678+0.j 0.70710678+0.j]
{'0': 0.4999999999999999, '1': 0.4999999999999999}
dict_keys(['0', '1'])


Remember: `wavefunction` is a debugging tool for small systems. The full wavefunction cannot feasibly be obtained from a real quantum processor.

## Multi-qubit basis enumeration

The `WavefunctionSimulator` enumerates bitstrings with qubit `0` as the **least significant bit (LSB), i.e., on the *right* end of the bitstring**:

| bitstring | qubit_(n-1) | … | qubit_2 | qubit_1 | qubit_0 |
|-----------|-------------|---|---------|---------|---------|
| 1…101     | 1           | … | 1       | 0       | 1       |
| 0…110     | 0           | … | 1       | 1       | 0       |

This is the opposite of the convention common in the literature, where the lowest-index qubit is on the left (**for example, CUDA-Q**). The vector representation assumes the canonical ordering of basis elements — for two qubits, `00, 01, 10, 11`. In Dirac notation, the tensor product of two degrees of freedom isn't always understood to have a fixed order, unlike the Kronecker product of matrices (same symbol, clearly not commutative). The order matters once you write coefficient vectors or matrices:

$$\ket{0}_0 \otimes \ket{1}_1 = \ket{1}_1 \otimes \ket{0}_0 = \ket{10}_{1,0} \equiv \begin{pmatrix} 0 \\ 0 \\ 1 \\ 0 \end{pmatrix}$$

This leads to subtle but important differences in the ordering of wavefunction and multi-qubit gate matrix coefficients. In this convention the matrix

$$U_{\rm CNOT(1,0)} \equiv \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \end{pmatrix}$$

corresponds to the Quil instruction `CNOT(1, 0)`, counter to how most of the field orders Kronecker products. `CNOT(0, 1)` is:

$$U_{\rm CNOT(0,1)} \equiv \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \\ 0 & 1 & 0 & 0 \end{pmatrix}$$

Rigetti's note on arXiv explains why this basis ordering was chosen.

### Try it: see the ordering in action

In [4]:
# X on qubit 0, with qubit 1 also in the program.
# Qubit 0 is the rightmost bit, so the state prints as |01>, not |10>.
wf2 = wf_sim.wavefunction(Program(X(0), I(1)))
print(wf2)
print(wf2.get_outcome_probs())

(1+0j)|01>
{'00': 0.0, '01': 1.0, '10': 0.0, '11': 0.0}


In [5]:
# Compare the CNOT matrices with pyQuil's own gate matrix definitions
from pyquil.simulation.matrices import CNOT as CNOT_matrix
print("pyQuil CNOT matrix (control = first qubit in textbook order):")
print(CNOT_matrix.real.astype(int))

# Prepare |1> on qubit 0 and apply CNOT(0, 1): qubit 1 should flip -> |11>
print(wf_sim.wavefunction(Program(X(0), CNOT(0, 1))))
# Prepare |1> on qubit 1 and apply CNOT(1, 0): qubit 0 should flip -> |11>
print(wf_sim.wavefunction(Program(X(1), CNOT(1, 0))))

pyQuil CNOT matrix (control = first qubit in textbook order):
[[1 0 0 0]
 [0 1 0 0]
 [0 0 0 1]
 [0 0 1 0]]
(1+0j)|11>
(1+0j)|11>


### Kill the servers after use in **0_Jupyter_Pykernel_from_Container_image** notebook